# SME-GPT OCR worker — Kaggle edition

Same OCR + embedding server as the Colab notebook (`ocr_only_mvp2.ipynb`), packaged to run as a
**background Kaggle job**: no browser tab has to stay open, and it runs for up to 12 hours.

## One-time setup
1. **Settings → Accelerator → GPU T4 x2.** Don't pick P100: current PyTorch no longer supports it.
2. **Settings → Internet → On.** This needs a phone-verified Kaggle account.
3. **Add-ons → Secrets.** Add both of these, and tick them for this notebook:
   - `NGROK_AUTH_TOKEN`: the authtoken of a **second** free ngrok account.
     A free ngrok account allows only one running tunnel, and Colab is already using yours.
   - `NGROK_DOMAIN`: that account's free static domain, from ngrok dashboard → Domains,
     e.g. `something-something.ngrok-free.app`.
4. Add the worker to the backend's `COLAB_OCR_URL` on the server, comma-separated after the Colab URL:
   `COLAB_OCR_URL=https://<colab-domain>,https://<kaggle-domain>`
   The backend tries them in order, so if one notebook stops, the next one takes over.

## Run it
- **Background (normal use):** **Save Version → Save & Run All (Commit)**. You can close the tab.
  It keeps serving until the 12-hour limit, or until you cancel it under **Your Work → the running version → Stop**.
- **Interactive (for debugging):** run all cells. The last cell blocks while serving.

Each run uses GPU quota (about 30 h/week on the free tier), so stop it when you no longer need it.


In [ ]:
# 1. Install. Torch and CUDA are preinstalled on Kaggle; pins match the Colab notebook.
!pip install -q "transformers==4.57.3" tokenizers accelerate python-multipart
!pip install -q "surya-ocr==0.17.1" sentence-transformers pyngrok


In [ ]:
# 2. Load Surya v1 models ONCE (in-process, GPU, no vLLM)
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

foundation = FoundationPredictor()
detector   = DetectionPredictor()
recognizer = RecognitionPredictor(foundation)

# Embedding model for the backend's RAG search (POST /embed below).
# REQUIRED when the backend runs with EMBEDDING_ENGINE=colab — do NOT delete.
# Must stay intfloat/multilingual-e5-small (384-dim): it has to match the
# vectors already stored in the ChunkEmbedding table.
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("intfloat/multilingual-e5-small")

print("Models loaded OK")


In [ ]:
# 3. OCR server matching the SME-GPT backend contract
#
# POST /ocr  (multipart form-data, multiple parts per field, one per page):
#     orig   - deskewed RGB image       (required)
#     p_img  - binarized variant        (optional)
#     m_img  - bilateral-filtered       (optional)
#   Returns: { success, engine, versions: {orig,P,M: {text, pages:[{page,text,text_lines}]}}, failures }
#   Each text_line: { text, confidence, polygon, bbox }
#
# POST /embed  (JSON {texts: [...], mode: "query"|"passage"}) -> {embeddings, dim}
#   Used by the backend when EMBEDDING_ENGINE=colab.
#
# GET /health -> { status, engine, mode }
#
# Surya/e5 inference is blocking, so it runs in worker threads: a long OCR
# job must not freeze the event loop, or every /embed call (one per chat
# question) would queue behind it and time out on the backend.

import io
from typing import List, Dict, Any

from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from starlette.concurrency import run_in_threadpool
from PIL import Image

NL = chr(10)

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_credentials=True,
    allow_methods=["*"], allow_headers=["*"],
)

SURYA_MODE = "surya-0.17.1 (v1 in-process/cuda, kaggle)"


def _ocr_images(images: List[Image.Image]) -> List[Dict[str, Any]]:
    """Run Surya v1 detection+recognition on a batch of page images.
    Returns [{text, text_lines:[{text,confidence,polygon,bbox}]}] (one per page)."""
    if not images:
        return []
    rgb = [im.convert("RGB") if im.mode != "RGB" else im for im in images]
    results = recognizer(rgb, det_predictor=detector)   # batched across pages

    pages = []
    for page_res in results:
        if hasattr(page_res, "model_dump"):
            pd = page_res.model_dump()
        elif hasattr(page_res, "dict"):
            pd = page_res.dict()
        else:
            pd = dict(page_res)

        lines = []
        for ln in pd.get("text_lines", []) or []:
            txt = (ln.get("text") or "").strip()
            if not txt:
                continue
            lines.append({
                "text"      : txt,
                "confidence": float(ln.get("confidence") or 0.0),
                "polygon"   : ln.get("polygon"),
                "bbox"      : ln.get("bbox"),
            })
        page_text = NL.join(l["text"] for l in lines).strip()
        pages.append({"text": page_text, "text_lines": lines})
    return pages


def _build_version(images: List[Image.Image], vname: str, failures: dict) -> dict:
    """OCR all pages of one variant -> {text, pages:[{page,text,text_lines}]}."""
    if not images:
        return {"text": "", "pages": []}
    try:
        page_results = _ocr_images(images)
    except Exception as exc:
        failures[vname] = str(exc)
        print("  [OCR] variant " + vname + " failed: " + str(exc), flush=True)
        return {"text": "", "pages": []}

    n = len(page_results)
    pages_out, parts = [], []
    for i, pr in enumerate(page_results, start=1):
        pages_out.append({"page": i, "text": pr["text"], "text_lines": pr["text_lines"]})
        if pr["text"]:
            parts.append(("=== PAGE " + str(i) + " ===" + NL + pr["text"]) if n > 1 else pr["text"])
    return {"text": (NL + NL).join(parts).strip(), "pages": pages_out}


async def _read_images(files: List[UploadFile]) -> List[Image.Image]:
    imgs = []
    for f in files:
        raw = await f.read()
        if raw:
            imgs.append(Image.open(io.BytesIO(raw)))
    return imgs


@app.get("/health")
def health():
    return {"status": "ok", "engine": "surya", "mode": SURYA_MODE}


@app.post("/ocr")
async def ocr(
    orig:  List[UploadFile] = File(default=[]),
    p_img: List[UploadFile] = File(default=[]),
    m_img: List[UploadFile] = File(default=[]),
):
    failures: Dict[str, Any] = {}

    orig_imgs = await _read_images(orig)
    if not orig_imgs:
        return {"success": False, "error": "No 'orig' images received.",
                "engine": "colab_surya_v1", "versions": {}, "failures": failures}

    p_imgs = await _read_images(p_img)
    m_imgs = await _read_images(m_img)
    print("[OCR] pages -> orig:" + str(len(orig_imgs)) + " P:" + str(len(p_imgs)) + " M:" + str(len(m_imgs)), flush=True)

    versions = {
        "orig": await run_in_threadpool(_build_version, orig_imgs, "orig", failures),
        "P":    await run_in_threadpool(_build_version, p_imgs,    "P",    failures),
        "M":    await run_in_threadpool(_build_version, m_imgs,    "M",    failures),
    }
    return {"success": True, "engine": "colab_surya_v1",
            "versions": versions, "failures": failures}


@app.post("/embed")
def embed(payload: dict):
    # Plain `def`: FastAPI runs it in its threadpool, off the event loop.
    texts = payload.get("texts", [])
    mode = payload.get("mode", "passage")
    if mode not in ("passage", "query"):
        mode = "passage"
    prefixed = [(mode + ": " + t) for t in texts]
    vecs = embedder.encode(prefixed, normalize_embeddings=True)
    return {"embeddings": vecs.tolist(), "dim": len(vecs[0]) if len(vecs) else 0}


In [ ]:
# 4. Serve: uvicorn in a background thread, the ngrok tunnel on the fixed domain,
#    then a keep-alive loop. This blocks the cell, which is what keeps a
#    background ("Save & Run All") job alive.
import subprocess, threading, time
import requests
import uvicorn
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
NGROK_TOKEN = secrets.get_secret("NGROK_AUTH_TOKEN")
NGROK_DOMAIN = secrets.get_secret("NGROK_DOMAIN").strip().removeprefix("https://").rstrip("/")
PUBLIC_URL = "https://" + NGROK_DOMAIN

# set_auth_token also downloads the ngrok binary on first use.
try:
    ngrok.set_auth_token(NGROK_TOKEN)
except Exception as e:
    print("ngrok token note:", e)

# The server runs in a worker thread: uvicorn must not install signal handlers there.
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None
threading.Thread(target=server.run, daemon=True).start()

for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
print("Local server is up.", flush=True)


def start_tunnel():
    # The ngrok binary is started directly: pyngrok's connect() is rejected by
    # current ngrok agents ("field ... not found in type config.Endpoint").
    return subprocess.Popen(
        [conf.get_default().ngrok_path, "http", "8000", "--url", PUBLIC_URL, "--log", "stdout"],
        stdout=open("/kaggle/working/ngrok.log", "w"), stderr=subprocess.STDOUT,
    )


tunnel = start_tunnel()
time.sleep(5)
print(open("/kaggle/working/ngrok.log").read()[-800:], flush=True)
print("=" * 50)
print(f"OCR worker is LIVE at: {PUBLIC_URL}")
print("=" * 50, flush=True)

started = time.time()
while True:
    time.sleep(300)
    if tunnel.poll() is not None:
        # ERR_NGROK_108 in the log means this ngrok account already has a tunnel
        # running elsewhere (e.g. the same account is in use on Colab).
        print("ngrok exited; restarting. Last log lines:", flush=True)
        print(open("/kaggle/working/ngrok.log").read()[-500:], flush=True)
        tunnel = start_tunnel()
    print(f"[keep-alive] serving for {int((time.time() - started) / 60)} min", flush=True)
